# Análise Estatística — Zika Vírus (SINAN 2018-2026)

Entrega 5 do projeto: sazonalidade, tendência por UF, previsão de casos (Prophet) e
agrupamento de municípios por perfil epidemiológico (K-Means).

Os resultados são exportados para `data/gold/` e serão consumidos pelo dashboard interativo.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.seasonal import STL
from prophet import Prophet

SILVER = Path("..") / "data" / "silver"
GOLD = Path("..") / "data" / "gold"
GOLD.mkdir(parents=True, exist_ok=True)

DATA_MIN = pl.date(2018, 1, 1)
DATA_MAX = pl.date(2026, 6, 10)

D:\Programacao\Faculdade\modelagem_estatistica\Projeto_dataSUS\zika-virus-epidemiological-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Preparação da base

Carrega as tabelas silver, decodifica `idade_codificada` (NU_IDADE_N) e monta a base de
notificações confirmadas (`CLASSI_FIN = 1`), removendo registros com data de primeiros
sintomas fora do período coberto pelo dataset (2018-2026).

In [2]:
notificacoes = pl.read_parquet(SILVER / "notificacoes_casos.parquet")
pacientes = pl.read_parquet(SILVER / "pacientes.parquet")
municipios = pl.read_parquet(SILVER / "municipios.parquet")
ufs = pl.read_parquet(SILVER / "ufs.parquet")

pacientes = pacientes.with_columns(
    idade_codificada_str=pl.col("idade_codificada").cast(pl.Utf8)
).with_columns(
    tipo=pl.col("idade_codificada_str").str.slice(0, 1),
    quantidade=pl.col("idade_codificada_str").str.slice(1).cast(pl.Float64),
).with_columns(
    idade_anos=pl.when(pl.col("tipo") == "4").then(pl.col("quantidade"))
    .when(pl.col("tipo") == "3").then(pl.col("quantidade") / 12.0)
    .when(pl.col("tipo") == "2").then(pl.col("quantidade") / 365.0)
    .when(pl.col("tipo") == "1").then(pl.col("quantidade") / 8760.0)
    .otherwise(None)
    .round(2)
).drop("idade_codificada_str", "tipo", "quantidade")

casos_confirmados = (
    notificacoes
    .filter(pl.col("classificacao_final") == "Confirmado")
    .filter(pl.col("data_primeiros_sintomas").is_between(DATA_MIN, DATA_MAX))
    .join(pacientes, on="paciente_id", how="left")
)

print(f"Notificações confirmadas no período: {casos_confirmados.height:,}")
casos_confirmados.head()

Notificações confirmadas no período: 28,202


notificacao_id,paciente_id,data_notificacao,semana_notificacao,unidade_id,regional_notificacao,data_primeiros_sintomas,semana_primeiros_sintomas,data_investigacao,classificacao_final,criterio_confirmacao,autoctonia,uf_provavel_infeccao,municipio_provavel_infeccao,pais_provavel_infeccao,doenca_trabalho,evolucao_caso,nduplic,fluxo_retorno,fluxo_recebido,idade_codificada,sexo,idade_gestacional,raca_cor,escolaridade,municipio_id,regional_residencia,pais_id,ocupacao,idade_anos
i64,i64,date,i16,i32,i64,date,i16,date,str,str,str,i16,i32,i16,str,str,bool,i16,i16,i16,str,str,str,str,i32,i64,i16,str,f64
55322,55322,2020-02-13,7,7107366,null,2020-02-09,7,2020-02-13,"""Confirmado""","""Clínico-epidemiológico""","""Sim""",33,3304557,1,"""Ignorado""","""Ignorado""",false,null,null,4031,"""Feminino""","""3º trimestre""","""Ignorado""","""Ignorado""",3304557,null,1,null,31.0
55323,55323,2020-02-13,7,2271443,null,2020-02-02,6,2020-02-13,"""Confirmado""","""Clínico-epidemiológico""","""Indeterminado""",null,null,null,"""Não""","""Cura""",false,null,null,4024,"""Masculino""","""Não se aplica""","""Parda""","""Ignorado""",3304557,null,1,null,24.0
55324,55324,2020-02-13,7,9112529,null,2020-02-08,6,2020-02-18,"""Confirmado""","""Clínico-epidemiológico""","""Sim""",51,5107040,1,"""Não""","""Cura""",false,null,null,4026,"""Feminino""","""Ignorado""","""Ignorado""","""Ignorado""",5107040,null,1,null,26.0
55325,55325,2020-02-13,7,9112529,null,2020-02-10,7,2020-02-18,"""Confirmado""","""Clínico-epidemiológico""","""Sim""",51,5107040,1,"""Não""","""Cura""",false,null,null,4007,"""Feminino""","""Não se aplica""","""Ignorado""","""Ignorado""",5107040,null,1,null,7.0
55336,55336,2020-02-13,7,2534398,null,2020-02-09,7,2020-02-13,"""Confirmado""","""Clínico-epidemiológico""","""Sim""",51,5103403,1,"""Não""","""Cura""",false,null,null,4018,"""Feminino""","""Ignorado""","""Ignorado""","""Ignorado""",5103403,null,1,null,18.0


## 2. Sazonalidade

Série temporal semanal de casos confirmados por data de primeiros sintomas
(`DT_SIN_PRI`), decomposta em tendência, sazonalidade e resíduo (STL, período de 52
semanas).

In [3]:
serie_semanal_pl = (
    casos_confirmados
    .with_columns(semana=pl.col("data_primeiros_sintomas").dt.truncate("1w"))
    .group_by("semana")
    .agg(pl.len().alias("casos"))
    .sort("semana")
)

serie_semanal = serie_semanal_pl.to_pandas().set_index("semana")
serie_semanal = serie_semanal.asfreq("W-MON", fill_value=0)

fig = px.line(serie_semanal, y="casos", title="Casos confirmados de Zika por semana epidemiológica (DT_SIN_PRI)")
fig.update_layout(xaxis_title="Semana", yaxis_title="Casos confirmados")
fig.show()

In [4]:
stl = STL(serie_semanal["casos"], period=52, robust=True)
resultado_stl = stl.fit()

decomposicao = pd.DataFrame({
    "casos": serie_semanal["casos"],
    "tendencia": resultado_stl.trend,
    "sazonalidade": resultado_stl.seasonal,
    "residuo": resultado_stl.resid,
})

fig = go.Figure()
for coluna in ["casos", "tendencia", "sazonalidade", "residuo"]:
    fig.add_trace(go.Scatter(x=decomposicao.index, y=decomposicao[coluna], name=coluna))
fig.update_layout(title="Decomposição STL da série semanal de casos confirmados", xaxis_title="Semana")
fig.show()

In [5]:
perfil_sazonal = (
    serie_semanal_pl
    .with_columns(semana_epi=pl.col("semana").dt.week())
    .group_by("semana_epi")
    .agg(pl.col("casos").mean().alias("media_casos"))
    .sort("semana_epi")
)

fig = px.bar(
    perfil_sazonal.to_pandas(), x="semana_epi", y="media_casos",
    title="Perfil sazonal médio — casos confirmados por semana epidemiológica (todos os anos)"
)
fig.update_layout(xaxis_title="Semana epidemiológica", yaxis_title="Média de casos confirmados")
fig.show()

## 3. Tendência por UF

Casos confirmados por UF provável de infecção (`COUFINF`) e ano de notificação,
com regressão linear `casos ~ ano` para medir a tendência (queda/crescimento) em
cada UF que possui pelo menos 4 anos de dados.

In [6]:
casos_uf_ano = (
    casos_confirmados
    .filter(pl.col("uf_provavel_infeccao").is_not_null())
    .with_columns(ano=pl.col("data_notificacao").dt.year())
    .group_by(["uf_provavel_infeccao", "ano"])
    .agg(pl.len().alias("total_casos"))
    .join(ufs, left_on="uf_provavel_infeccao", right_on="uf_id", how="left")
    .sort(["sigla", "ano"])
)

casos_uf_ano.head(10)

uf_provavel_infeccao,ano,total_casos,sigla,nome
i16,i32,u32,str,str
12,2018,87,"""AC""","""Acre"""
12,2019,28,"""AC""","""Acre"""
12,2020,22,"""AC""","""Acre"""
12,2021,258,"""AC""","""Acre"""
12,2022,2,"""AC""","""Acre"""
12,2023,92,"""AC""","""Acre"""
12,2024,41,"""AC""","""Acre"""
12,2025,12,"""AC""","""Acre"""
27,2018,116,"""AL""","""Alagoas"""


In [7]:
tendencias = []
for uf, grupo in casos_uf_ano.to_pandas().groupby("sigla"):
    if grupo["ano"].nunique() < 4:
        continue
    slope, intercept, r_value, p_value, std_err = stats.linregress(grupo["ano"], grupo["total_casos"])
    tendencias.append({
        "uf": uf,
        "anos_com_dados": grupo["ano"].nunique(),
        "total_casos": grupo["total_casos"].sum(),
        "tendencia_casos_ano": slope,
        "r2": r_value ** 2,
        "p_value": p_value,
    })

tendencia_uf = pd.DataFrame(tendencias).sort_values("tendencia_casos_ano", ascending=False)
tendencia_uf

,uf,anos_com_dados,total_casos,tendencia_casos_ano,r2,p_value
12,MT,9,3181,41.133333,0.116912,0.367782
22,RS,7,122,0.440789,0.003253,0.903328
21,RR,9,147,-0.016667,0.000007,0.994605
23,SC,5,9,-0.060345,0.030172,0.779953
17,PR,6,18,-0.359155,0.305282,0.255553
9,MA,9,607,-0.416667,0.000445,0.957058
16,PI,8,168,-1.047619,0.079750,0.497979
3,AP,9,136,-1.183333,0.046966,0.575426
20,RO,9,395,-1.933333,0.073173,0.481438
6,DF,6,72,-3.321888,0.219306,0.348899


In [8]:
fig = px.bar(
    tendencia_uf, x="uf", y="tendencia_casos_ano", color="tendencia_casos_ano",
    color_continuous_scale="RdYlGn_r",
    title="Tendência anual de casos confirmados por UF provável de infecção (inclinação da regressão linear)"
)
fig.update_layout(xaxis_title="UF", yaxis_title="Variação média de casos por ano")
fig.show()

top_ufs = tendencia_uf.nlargest(5, "total_casos")["uf"].tolist()
fig = px.line(
    casos_uf_ano.filter(pl.col("sigla").is_in(top_ufs)).to_pandas(),
    x="ano", y="total_casos", color="sigla", markers=True,
    title=f"Série anual de casos confirmados — UFs com mais casos ({', '.join(top_ufs)})"
)
fig.show()

## 4. Previsão de casos (Prophet)

Usa a série semanal de casos confirmados (item 2) para treinar um modelo Prophet
com sazonalidade anual e prever as próximas 12 semanas. O desempenho é avaliado
retreinando o modelo sem as últimas 12 semanas observadas e comparando a previsão
com os valores reais (holdout).

In [9]:
serie_prophet = serie_semanal.reset_index().rename(columns={"semana": "ds", "casos": "y"})

HORIZONTE = 12
treino = serie_prophet.iloc[:-HORIZONTE]
teste = serie_prophet.iloc[-HORIZONTE:]

modelo_holdout = Prophet(weekly_seasonality=False, yearly_seasonality=True, seasonality_mode="additive")
modelo_holdout.fit(treino)

futuro_holdout = modelo_holdout.make_future_dataframe(periods=HORIZONTE, freq="W-MON")
previsao_holdout = modelo_holdout.predict(futuro_holdout)

avaliacao = previsao_holdout.tail(HORIZONTE).set_index("ds")[["yhat"]].join(teste.set_index("ds")[["y"]])
mae = mean_absolute_error(avaliacao["y"], avaliacao["yhat"])
rmse = mean_squared_error(avaliacao["y"], avaliacao["yhat"]) ** 0.5

print(f"MAE (holdout de {HORIZONTE} semanas): {mae:.2f}")
print(f"RMSE (holdout de {HORIZONTE} semanas): {rmse:.2f}")

21:31:55 - cmdstanpy - INFO - Chain [1] start processing


21:31:56 - cmdstanpy - INFO - Chain [1] done processing


MAE (holdout de 12 semanas): 41.01
RMSE (holdout de 12 semanas): 43.39


In [10]:
modelo_final = Prophet(weekly_seasonality=False, yearly_seasonality=True, seasonality_mode="additive")
modelo_final.fit(serie_prophet)

futuro = modelo_final.make_future_dataframe(periods=HORIZONTE, freq="W-MON")
previsao = modelo_final.predict(futuro)

fig = go.Figure()
fig.add_trace(go.Scatter(x=serie_prophet["ds"], y=serie_prophet["y"], name="Casos observados", mode="lines"))
fig.add_trace(go.Scatter(x=previsao["ds"], y=previsao["yhat"], name="Previsão", mode="lines"))
fig.add_trace(go.Scatter(
    x=pd.concat([previsao["ds"], previsao["ds"][::-1]]),
    y=pd.concat([previsao["yhat_upper"], previsao["yhat_lower"][::-1]]),
    fill="toself", fillcolor="rgba(0,100,80,0.2)", line=dict(color="rgba(255,255,255,0)"),
    name="Intervalo de confiança", showlegend=True,
))
fig.update_layout(title=f"Previsão de casos confirmados de Zika — próximas {HORIZONTE} semanas", xaxis_title="Semana", yaxis_title="Casos")
fig.show()

21:31:56 - cmdstanpy - INFO - Chain [1] start processing


21:31:56 - cmdstanpy - INFO - Chain [1] done processing


## 5. Agrupamento de municípios (K-Means)

Para cada município (residência do paciente, `ID_MN_RESI`) com pelo menos 10 casos
confirmados, calcula-se um perfil epidemiológico:

- `total_confirmados`: volume de casos confirmados
- `taxa_confirmacao`: proporção de notificações confirmadas sobre o total notificado
- `idade_media`: idade média dos casos confirmados
- `pct_gestantes`: proporção de casos confirmados em gestantes
- `pct_obito`: proporção de casos confirmados que evoluíram a óbito pelo agravo
- `pct_autoctone`: proporção de casos confirmados classificados como autóctones

Os municípios são então agrupados via K-Means em perfis epidemiológicos.

In [11]:
base_municipio = notificacoes.join(pacientes, on="paciente_id", how="left")

total_notificados = (
    base_municipio
    .filter(pl.col("municipio_id").is_not_null())
    .group_by("municipio_id")
    .agg(pl.len().alias("total_notificados"))
)

base_confirmados = base_municipio.filter(
    (pl.col("classificacao_final") == "Confirmado") & pl.col("municipio_id").is_not_null()
)

perfil_municipio = (
    base_confirmados
    .group_by("municipio_id")
    .agg(
        total_confirmados=pl.len(),
        idade_media=pl.col("idade_anos").mean(),
        pct_gestantes=(pl.col("idade_gestacional").is_in(["1º trimestre", "2º trimestre", "3º trimestre"]).sum() / pl.len()),
        pct_obito=((pl.col("evolucao_caso") == "Óbito pelo agravo").sum() / pl.len()),
        pct_autoctone=((pl.col("autoctonia") == "Sim").sum() / pl.len()),
    )
    .join(total_notificados, on="municipio_id", how="left")
    .with_columns(taxa_confirmacao=pl.col("total_confirmados") / pl.col("total_notificados"))
    .filter(pl.col("total_confirmados") >= 10)
    .join(municipios, on="municipio_id", how="left")
    .join(ufs, on="uf_id", how="left")
    .rename({"nome_right": "uf_nome"})
)

print(f"Municípios elegíveis para o agrupamento: {perfil_municipio.height}")
perfil_municipio.head()

Municípios elegíveis para o agrupamento: 437


municipio_id,total_confirmados,idade_media,pct_gestantes,pct_obito,pct_autoctone,total_notificados,taxa_confirmacao,nome,uf_id,sigla,uf_nome
i32,u32,f64,f64,f64,f64,u32,f64,str,i16,str,str
5107297,15,58.066667,0.0,0.0,0.866667,21,0.714286,"""São José do Povo""",51,"""MT""","""Mato Grosso"""
2705002,11,21.174545,0.272727,0.0,1.0,20,0.55,"""Mata Grande""",27,"""AL""","""Alagoas"""
2602902,19,31.578947,0.052632,0.0,0.578947,181,0.104972,"""Cabo de Santo Agostinho""",26,"""PE""","""Pernambuco"""
2515203,11,29.636364,0.181818,0.0,1.0,14,0.785714,"""São Sebastião do Umbuzeiro""",25,"""PB""","""Paraíba"""
2312908,12,28.465,0.0,0.0,0.583333,1241,0.00967,"""Sobral""",23,"""CE""","""Ceará"""


In [12]:
FEATURES = ["total_confirmados", "taxa_confirmacao", "idade_media", "pct_gestantes", "pct_obito", "pct_autoctone"]

perfil_pd = perfil_municipio.to_pandas()
X = perfil_pd[FEATURES].fillna(0)
X_scaled = StandardScaler().fit_transform(X)

inercias, silhuetas = [], []
ks = range(2, 9)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_scaled, km.labels_))

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(ks), y=inercias, name="Inércia", yaxis="y1"))
fig.add_trace(go.Scatter(x=list(ks), y=silhuetas, name="Silhueta", yaxis="y2"))
fig.update_layout(
    title="Escolha do número de clusters (K-Means)",
    xaxis_title="k",
    yaxis=dict(title="Inércia"),
    yaxis2=dict(title="Silhueta", overlaying="y", side="right"),
)
fig.show()

In [13]:
K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10).fit(X_scaled)
perfil_pd["cluster"] = kmeans.labels_

perfil_clusters = perfil_pd.groupby("cluster")[FEATURES].mean().round(3)
perfil_clusters["n_municipios"] = perfil_pd.groupby("cluster").size()
perfil_clusters

,total_confirmados,taxa_confirmacao,idade_media,pct_gestantes,pct_obito,pct_autoctone,n_municipios
cluster,,,,,,,
0,545.500,0.436,30.656,0.059,0.01,0.737,12
1,43.083,0.162,31.763,0.048,0.00,0.910,216
2,46.163,0.716,31.978,0.052,0.00,0.921,135
3,37.554,0.243,29.853,0.124,0.00,0.329,74


In [14]:
coords = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
perfil_pd["pca_1"], perfil_pd["pca_2"] = coords[:, 0], coords[:, 1]

fig = px.scatter(
    perfil_pd, x="pca_1", y="pca_2", color=perfil_pd["cluster"].astype(str),
    hover_data=["nome", "sigla", "total_confirmados", "taxa_confirmacao", "idade_media", "pct_gestantes", "pct_obito", "pct_autoctone"],
    title=f"Municípios agrupados por perfil epidemiológico (K-Means, k={K})"
)
fig.show()

## 6. Exportação dos resultados para o dashboard

In [15]:
decomposicao.reset_index().to_parquet(GOLD / "sazonalidade_semanal.parquet", index=False)
perfil_sazonal.write_parquet(GOLD / "perfil_sazonal_semana_epi.parquet")
tendencia_uf.to_parquet(GOLD / "tendencia_casos_uf.parquet", index=False)
previsao[["ds", "yhat", "yhat_lower", "yhat_upper"]].to_parquet(GOLD / "previsao_casos_semanal.parquet", index=False)
perfil_pd.drop(columns=["pca_1", "pca_2"]).to_parquet(GOLD / "clusters_municipios.parquet", index=False)

print("Arquivos exportados em", GOLD.resolve())
for arquivo in sorted(GOLD.glob("*.parquet")):
    print("-", arquivo.name)

Arquivos exportados em D:\Programacao\Faculdade\modelagem_estatistica\Projeto_dataSUS\zika-virus-epidemiological-analysis\data\gold
- clusters_municipios.parquet
- perfil_sazonal_semana_epi.parquet
- previsao_casos_semanal.parquet
- sazonalidade_semanal.parquet
- tendencia_casos_uf.parquet
